In [7]:
from google.colab import files
from IPython.display import display
import pandas as pd
import numpy as np
import re
import unicodedata

INPUT_FILE = "/content/dkktdl.csv"
OUTPUT_FILE = "/content/dkktdl_sach.csv"

df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig", dtype=str)

print(f"Dữ liệu ban đầu: {df.shape[0]} dòng, {df.shape[1]} cột")
display(df.head())

Dữ liệu ban đầu: 50 dòng, 13 cột


,Cafe_ID,Ten_quan,Dia_chi,Vi_do,Kinh_do,Gia_trung_binh,Diem_danh_gia(Rating),So_luong_review,Co_wifi,Co_o_cam_dien,Do_yen_tinh,Phu_hop_hoc_nhom,Khoang_cach_HUIT_km
0,CF001,Focus Space,"49B Tân Thành, Tân Phú",10.786906,106.638381,45000,5.0,175,1,1,4,1,2.437118
1,CF002,Cycle Lab Coffe,"149A Trịnh Đình Trọng, phường, Tân Phú",10.776064,106.641006,44000,4.7,180,1,1,3,1,3.65818
2,CF003,K-BOX Coffee 24/24,"119 Lê Trọng Tấn, Tân Sơn Nhì",10.805926,106.628439,42000,4.0,243,1,1,4,0,0.094667
3,CF004,MeetCha,"325 Lê Trọng Tấn, Tân Sơn Nhì",10.807929,106.620355,32500,5.0,81,1,1,4,1,0.932456
4,CF005,The Banned Coffee,"Tân Phú, 109 Tân Thắng, Tân Sơn Nhì",10.800497,106.617573,62000,3.7,240,1,0,2,0,1.406933


In [8]:
# CHUẨN HÓA
df.columns = df.columns.str.strip()

def chuan_hoa_chuoi(value):
    if pd.isna(value):
        return pd.NA

    value = unicodedata.normalize("NFC", str(value))
    value = " ".join(value.split())

    if value.lower() in ["", "na", "n/a", "nan", "null", "none"]:
        return pd.NA

    return value

for col in df.columns:
    df[col] = df[col].map(chuan_hoa_chuoi)

so_dong_ban_dau = len(df)
so_o_thieu_ban_dau = int(df.isna().sum().sum())


# CHUYỂN GIÁ TIỀN VÀ LƯỢT ĐÁNH GIÁ SANG SỐ
def chuyen_so(value):
    if pd.isna(value):
        return np.nan

    s = str(value).lower().strip()
    he_so = 1

    if re.search(r"(k|nghìn|ngàn)\s*$", s):
        he_so = 1000
        s = re.sub(r"(k|nghìn|ngàn)\s*$", "", s)

    s = re.sub(r"(vnđ|vnd|đồng|đ|₫)", "", s)
    s = re.sub(r"\s+", "", s)


    if he_so == 1 and re.fullmatch(r"\d{1,3}([.,]\d{3})+", s):
        s = re.sub(r"[.,]", "", s)
    else:
        s = s.replace(",", ".")

    try:
        return float(s) * he_so
    except ValueError:
        return np.nan

for col in ["Gia_trung_binh", "So_luong_review"]:
    df[col] = df[col].map(chuyen_so)


# CHUẨN HÓA GPS VÀ CÁC CỘT SỐ CÒN LẠI
cot_so_con_lai = [
    "Vi_do", "Kinh_do", "Diem_danh_gia(Rating)",
    "Co_wifi", "Co_o_cam_dien", "Do_yen_tinh",
    "Phu_hop_hoc_nhom"
]

for col in cot_so_con_lai:
    df[col] = pd.to_numeric(
        df[col].astype("string").str.replace(",", ".", regex=False),
        errors="coerce"
    )

cot_nguyen = [
    "Gia_trung_binh", "So_luong_review", "Co_wifi",
    "Co_o_cam_dien", "Do_yen_tinh", "Phu_hop_hoc_nhom"
]
cot_so = ["Vi_do", "Kinh_do", "Diem_danh_gia(Rating)"] + cot_nguyen

df[cot_so] = df[cot_so].replace([np.inf, -np.inf], np.nan)

# Giá trị ngoài phạm vi hoặc sai kiểu được xem là thiếu
mien_hop_le = {
    "Vi_do": (-90, 90),
    "Kinh_do": (-180, 180),
    "Diem_danh_gia(Rating)": (0, 5),
    "Do_yen_tinh": (1, 5),
    "Co_wifi": (0, 1),
    "Co_o_cam_dien": (0, 1),
    "Phu_hop_hoc_nhom": (0, 1),
}

for col, (thap, cao) in mien_hop_le.items():
    df[col] = df[col].where(df[col].between(thap, cao))

df["Gia_trung_binh"] = df["Gia_trung_binh"].where(
    df["Gia_trung_binh"] > 0
)
df["So_luong_review"] = df["So_luong_review"].where(
    df["So_luong_review"] >= 0
)

for col in cot_nguyen:
    df[col] = df[col].where(df[col] % 1 == 0)


# XÓA DÒNG TRÙNG HOÀN TOÀN
so_dong_trung = int(df.duplicated().sum())
df = df.drop_duplicates().copy()


# ĐIỀN GIÁ TRỊ THIẾU
so_o_da_dien = {}

# Các cột định lượng: điền trung vị
for col in ["Gia_trung_binh", "So_luong_review", "Diem_danh_gia(Rating)"]:
    truoc = int(df[col].isna().sum())

    if df[col].notna().any():
        gia_tri_dien = df[col].median()

        if col in cot_nguyen:
            gia_tri_dien = int(np.floor(gia_tri_dien + 0.5))

        df[col] = df[col].fillna(gia_tri_dien)

    so_o_da_dien[col] = truoc - int(df[col].isna().sum())

# Các cột phân loại/thứ bậc: điền giá trị phổ biến nhất
for col in ["Co_wifi", "Co_o_cam_dien", "Do_yen_tinh", "Phu_hop_hoc_nhom"]:
    truoc = int(df[col].isna().sum())
    mode = df[col].mode()

    if not mode.empty:
        df[col] = df[col].fillna(mode.iloc[0])

    so_o_da_dien[col] = truoc - int(df[col].isna().sum())

# Không tự đoán GPS, mã, tên hoặc địa chỉ quán.
# Dòng còn thiếu được tách riêng để bổ sung.
mask_can_bo_sung = df.isna().any(axis=1)
df_can_bo_sung = df.loc[mask_can_bo_sung].copy()
df = df.loc[~mask_can_bo_sung].copy()

if df["Cafe_ID"].duplicated().any():
    raise ValueError(
        "Có Cafe_ID trùng nhưng nội dung khác nhau. Cần kiểm tra thủ công."
    )

for col in cot_nguyen:
    df[col] = df[col].astype("int64")


# TÍNH KHOẢNG CÁCH HAVERSINE
HUIT_LAT = 10.8067
HUIT_LON = 106.6288

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0088

    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

df["Khoang_cach_HUIT_km"] = haversine(
    HUIT_LAT,
    HUIT_LON,
    df["Vi_do"].to_numpy(dtype=float),
    df["Kinh_do"].to_numpy(dtype=float)
).round(6)

df = df.reset_index(drop=True)


# KIỂM TRA KẾT QUẢ
print(f"Số dòng ban đầu: {so_dong_ban_dau}")
print(f"Số ô thiếu ban đầu: {so_o_thieu_ban_dau}")
print(f"Số dòng trùng đã xóa: {so_dong_trung}")
print(f"Số ô đã điền: {sum(so_o_da_dien.values())}")
print(f"Số dòng cần bổ sung thủ công: {len(df_can_bo_sung)}")
print(f"Dữ liệu sạch: {df.shape[0]} dòng, {df.shape[1]} cột")

display(df.head(10))

Số dòng ban đầu: 50
Số ô thiếu ban đầu: 0
Số dòng trùng đã xóa: 0
Số ô đã điền: 0
Số dòng cần bổ sung thủ công: 0
Dữ liệu sạch: 50 dòng, 13 cột


,Cafe_ID,Ten_quan,Dia_chi,Vi_do,Kinh_do,Gia_trung_binh,Diem_danh_gia(Rating),So_luong_review,Co_wifi,Co_o_cam_dien,Do_yen_tinh,Phu_hop_hoc_nhom,Khoang_cach_HUIT_km
0,CF001,Focus Space,"49B Tân Thành, Tân Phú",10.786906,106.638381,45000,5.0,175,1,1,4,1,2.437118
1,CF002,Cycle Lab Coffe,"149A Trịnh Đình Trọng, phường, Tân Phú",10.776064,106.641006,44000,4.7,180,1,1,3,1,3.658180
2,CF003,K-BOX Coffee 24/24,"119 Lê Trọng Tấn, Tân Sơn Nhì",10.805926,106.628439,42000,4.0,243,1,1,4,0,0.094667
3,CF004,MeetCha,"325 Lê Trọng Tấn, Tân Sơn Nhì",10.807929,106.620355,32500,5.0,81,1,1,4,1,0.932456
4,CF005,The Banned Coffee,"Tân Phú, 109 Tân Thắng, Tân Sơn Nhì",10.800497,106.617573,62000,3.7,240,1,0,2,0,1.406933
5,CF006,Homebrew Café,"Tân Thắng 75-77 Đường Tân Thắng, Sơn Kỳ, Tân Phú",10.804691,106.622421,49000,4.8,969,1,1,3,1,0.731673
6,CF007,Hiên Nhà Coffee,"15 Đ. Vườn Lài, Phú Thọ Hòa, Tân Phú",10.78699,106.635592,42500,4.5,285,1,0,1,0,2.313811
7,CF008,New Day Cafe - Eats,"53Q Đ. Kênh Tân Hóa, Tân Phú",10.765794,106.635396,39500,4.0,171,0,1,3,1,4.605254
8,CF009,W Station - Study & Chill Café,"267G Trịnh Đình Trọng, Tân Phú",10.776604,106.637175,37500,4.9,180,1,1,4,1,3.469306
9,CF010,THREE O'CLOCK,"202-204 Trương Vĩnh Ký, Tân Sơn Nhì",10.794536,106.631258,56000,4.5,579,1,1,4,1,1.378965


In [9]:
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"Đã lưu: {OUTPUT_FILE}")

# Nếu có dòng chưa đủ dữ liệu thì lưu riêng để kiểm tra
if not df_can_bo_sung.empty:
    REVIEW_FILE = "dlktdl_can_bo_sung.csv"
    df_can_bo_sung.to_csv(
        REVIEW_FILE, index=False, encoding="utf-8-sig"
    )
    print(f"Các dòng cần bổ sung được lưu tại: {REVIEW_FILE}")
    display(df_can_bo_sung)

files.download(OUTPUT_FILE)

Đã lưu: /content/dkktdl_sach.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>